In [ ]:
from board import Loc, Locality, Target, Cell, Node, Board, Transformer, transform, parse, DIGITS, POS9

In [ ]:
from collections.abc import Generator


def iter_layer(board: Board, digit: int) -> Generator[Target]:
    for node in board:
        if digit in node.cell:
            yield Target(node.loc, digit)

## A puzzle


In [ ]:
puzzle = parse("""
.8.....52
.......87
....98...
4...3.6..
.2.7.....
.........
6..8.2...
...5.91..
9........
""")

## UI


In [ ]:
from typing import Iterable, Any
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from canvas import SudokuCanvas

In [ ]:
debug_view = w.Output()

LINKSTYLES = {"HLink": "HARD", "SLink": "SOFT"}
column_layout = w.Layout(width="auto", height="100%", flex_flow="column", align_items="stretch")


canvas = SudokuCanvas()
# selecting_layers = w.SelectMultiple(
#     options=(0,) + DIGITS,
#     value=[0],
#     layout=column_layout,
# )
selecting_targets = w.SelectMultiple(
    options=[],
    value=[],
    layout=column_layout,
    style=dict(description_width="0"),
)
selecting_links = w.SelectMultiple(
    options=[],
    value=[],
    layout=column_layout,
)
selecting_chains = w.Select(
    options=[],
    value=None,
    layout=column_layout,
)


def deselect(widget):
    widget.value = () if isinstance(widget, w.SelectMultiple) else None


@canvas.on_client_ready
def init_canvas():
    canvas[2].global_alpha = 0.5
    canvas.draw_grid()


@canvas.on_mouse_up
def on_canvas_click(x, y):
    targ = canvas.map_target(x, y)

    if not any(t == targ for _, t in selecting_targets.options):
        return

    if targ in selecting_targets.value:
        if selecting_targets.value is not None:
            selecting_targets.value = [v for v in selecting_targets.value if v != targ]
    else:
        if selecting_targets.value is not None:
            selecting_targets.value += (targ,)
        else:
            selecting_targets.value = (targ,)


# @selecting_layers.observe
# def on_select_layer(change):
#     selected = change.new
#     canvas.clear_highlights()
#     if len(selected) == 0:
#         return
#     with hold_canvas():
#         for digit in selected:
#             for target in iter_layer(puzzle, digit):
#                 canvas.highlight_target(target)


@selecting_targets.observe
def on_select_target(change):
    if change.name != "value":
        return

    print("targets", change)

    if len(change.old):
        with hold_canvas():
            canvas.clear_highlights()

    selected = change.new
    if len(selected):
        deselect(selecting_links)
        deselect(selecting_chains)

        with hold_canvas():
            for target in selected:
                canvas.highlight_target(target)


@selecting_links.observe
def on_select_link(change):
    if change.name != "value":
        return

    if len(change.old):
        with hold_canvas():
            canvas.clear_highlights()

    selected = change.new
    if len(selected):
        deselect(selecting_targets)
        deselect(selecting_chains)
        with hold_canvas():
            for lnk in selected:
                canvas.highlight_link(lnk, style=LINKSTYLES[lnk.__class__.__name__])
            for lnk in selected:
                canvas.highlight_target(lnk[0])
                canvas.highlight_target(lnk[1])


@selecting_chains.observe
def on_select_chain(change):
    if change.name != "value":
        return

    if change.old is not None:
        with hold_canvas():
            canvas.clear_highlights()

    selected = change.new
    if selected is not None:
        deselect(selecting_targets)
        deselect(selecting_links)
        with hold_canvas():
            for lnk in selected:
                canvas.highlight_link(lnk, style=LINKSTYLES[lnk.__class__.__name__])
            for trg in selected.anchors():
                canvas.highlight_target(trg)


def redraw_board():
    canvas.clear_highlights()
    canvas.draw_board(puzzle)


def load_options(widget, objects: Iterable[Any]):
    widget.options = [(str(obj), obj) for obj in objects]


w.HBox(
    [
        canvas,
        # w.VBox([w.Label("Layers"), selecting_layers]),
        w.VBox([w.Label("Targets"), selecting_targets]),
        w.VBox([w.Label("Links"), selecting_links]),
        w.VBox([w.Label("Chains"), selecting_chains]),
    ],
    layout=dict(justify_content="flex-start", align_items="stretch"),
)

In [ ]:
debug_view

In [ ]:
redraw_board()

In [ ]:
# using future vars
load_options(selecting_targets, anchors)
load_options(selecting_links, hardlinks)
load_options(selecting_chains, chains)

## Solving

kinda


In [ ]:
from typing import Iterable, Self
from itertools import chain as iterchain
from collections import deque

In [ ]:
def fillempty(board: Board, node: Node):
    if node.cell.is_empty:
        return Node(node.loc, Cell(DIGITS))
    else:
        return node

### Basic


In [ ]:
def cleanup(board: Board, node: Node) -> Node:
    """Clean up direct contradictions in localities"""

    def finals(loc: Locality):
        cells = [n.cell for n in board.slice(iter(loc))]
        return set(c.final for c in cells if c.is_final)

    if node.cell.is_final:
        return node
    blkfinals = finals(Locality(node.loc.blk, ..., ...))
    rowfinals = finals(Locality(..., node.loc.row, ...))
    colfinals = finals(Locality(..., ..., node.loc.col))
    allfinals = blkfinals | rowfinals | colfinals
    return Node(node.loc, Cell(set(node.cell) - allfinals))


In [ ]:
puzzle = transform(puzzle, fillempty)
puzzle = transform(puzzle, cleanup)

## Links


In [ ]:
class Link(tuple[Target, Target]):
    """Ordered set of targets
    (with symmetric equality)
    """

    def __str__(self):
        return f"{self[0]} ~ {self[1]}"

    def strtail(self):
        return f" ~ {self[1]}"

    def __repr__(self):
        return f"{self.__class__.__name__}(({self[0]!r}, {self[1]!r},))"

    def reversed(self):
        return self.__class__((self[1], self[0]))

    def __hash__(self):
        # symmetric hash
        return tuple.__hash__(self) + tuple.__hash__(self.reversed())

    def __eq__(self, other):
        # symmetric equality
        return hash(self) == hash(other)


class HLink(Link):
    """Hard link, XOR relation"""

    def __str__(self):
        return f"{self[0]}⟺{self[1]}"

    def strtail(self):
        return f"⟺{self[1]}"


class SLink(Link):
    """Soft link, NAND relation"""

    def __str__(self):
        return f"{self[0]}⟷{self[1]}"

    def strtail(self):
        return f"⟷{self[1]}"

### hard links

Represent XOR relation

Criteria (for signular targets):

- only 2 drafts of same digit in a locality
- only 2 drafts in a cell


In [ ]:
def find_hardlinks(board: Board) -> Generator[HLink]:

    def scan_cell(loc: Loc):
        node = board.get(loc)
        if len(node.cell) == 2:
            d1, d2 = node.cell
            yield HLink((
                Target(node.loc, d1),
                Target(node.loc, d2),
            ))

    def scan_locality(loc: Locality):
        nodes = board.slice(iter(loc))
        for d in DIGITS:
            sublayer = tuple(n for n in nodes if d in n.cell)
            if len(sublayer) == 2:
                n1, n2 = sublayer
                yield HLink((
                    Target(n1.loc, d),
                    Target(n2.loc, d),
                ))

    for node in board:
        yield from scan_cell(node.loc)
    for i in POS9:
        yield from scan_locality(Locality(i, ..., ...))
        yield from scan_locality(Locality(..., i, ...))
        yield from scan_locality(Locality(..., ..., i))


In [ ]:
hardlinks = set[HLink](find_hardlinks(puzzle))
anchors = set(iterchain.from_iterable(hardlinks))

In [ ]:
for lnk in hardlinks:
    print(lnk)

### soft links

Represent NAND relation

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts in a cell


In [ ]:
def check_nand(t1: Target, t2: Target):
    l1 = t1.loc
    l2 = t2.loc
    if t1.seg == t2.seg:
        return l1.blk == l2.blk or l1.row == l2.row or l1.col == l2.col
    else:
        return l1 == l2

### chains

Alterating link chains: (-xor-nand-)^n

(A-xor-B-nand-)^n-xor-D => All (X nand A) and (X nand D) can be eliminated


In [ ]:
class Chain(tuple[Link, ...]):
    @classmethod
    def init(cls, link: Link):
        return cls((link,))

    def __str__(self):
        return "".join([str(self[0])] + [lnk.strtail() for lnk in self[1:]])

    def __add__(self, other: Self):
        assert self[-1][-1] == other[0][0]
        return Chain(tuple(self) + tuple(other))

    def anchors(self) -> Iterable[Target]:
        """All anchor points in the chain"""
        return (self[0][0], *(lnk[1] for lnk in self))

    def ends(self):
        return (self[0][0], self[-1][1])

    def __hash__(self):
        """Hashing by unordered anchors"""
        return hash(frozenset(self.anchors()))

    def __eq__(self, other: Self):
        return hash(self) == hash(other)


def check_alc(chain: Chain):
    """Check if it's ALC"""
    return len(chain) > 2 and len(chain) % 3 == 0 and isinstance(chain[0], HLink) and isinstance(chain[-1], HLink)


def check_goal(chain: Chain):
    """Check if it's ALC and it's ends are softadjacent"""
    end0 = chain[0][0]
    end1 = chain[-1][-1]
    return check_alc(chain) and check_nand(end0, end1)

In [ ]:
# global hardlinks


def check_expansion(last: HLink, other: HLink) -> tuple[SLink, HLink] | None:
    front = last[1]
    if front in other:
        return None
    if check_nand(front, other[0]):
        return SLink((front, other[0])), other
    if check_nand(front, other[1]):
        return SLink((front, other[1])), other.reversed()


def expand_alc(chain: Chain) -> Generator[Chain]:
    """Append all possible -NAND-XOR links to end of chain"""
    last: HLink = chain[-1]  # type: ignore
    for other in hardlinks:
        if other not in chain:
            expansion = check_expansion(last, other)
            if expansion is not None and expansion[0] not in chain:
                yield chain + Chain(expansion)


def search_chains(initial: Link | None = None):
    """Seach for all ALC chains
    Depth-first search
    """
    if initial is not None:
        front = deque[Chain]((Chain.init(initial),))
    else:
        front = deque[Chain](Chain.init(lnk) for lnk in hardlinks)
    explored = set[Chain]()
    while front:
        chain = front.pop()
        if check_alc(chain):
            yield chain
        # if check_goal(chain):
        #     yield chain
        explored.add(chain)
        front.extend(child for child in expand_alc(chain) if child not in explored and child not in front)

In [ ]:
chains = set(search_chains())

In [ ]:
for ch in chains:
    print(ch)